# Analyse exploratoire — [DATASET NAME]
EDA préalable à la recherche automatisée de modèles ML.

## 1. Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 100

## 2. Configuration

In [ ]:
# ── À adapter pour chaque dataset ──────────────────────────────────────────
DATA_PATH   = "../data/00_raw/DATASET.csv"
TARGET_COL  = "target"        # colonne cible (classification ou régression)
TASK        = "classification" # "classification" | "regression"
DROP_COLS   = []              # colonnes à ignorer (ex: id, index)
# ───────────────────────────────────────────────────────────────────────────

## 3. Chargement des données

In [ ]:
df = pd.read_csv(DATA_PATH)
if DROP_COLS:
    df.drop(columns=DROP_COLS, inplace=True)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
features = [c for c in df.columns if c != TARGET_COL]
num_features = df[features].select_dtypes(include="number").columns.tolist()
cat_features = df[features].select_dtypes(exclude="number").columns.tolist()
print(f"Features numériques ({len(num_features)}) : {num_features}")
print(f"Features catégorielles ({len(cat_features)}) : {cat_features}")
df[num_features].describe()

## 4. Qualité des données

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print("Valeurs manquantes :")
display(pd.DataFrame({"count": missing, "%": missing_pct})[missing > 0])

print(f"\nDoublons : {df.duplicated().sum()}")

In [ ]:
if TASK == "classification":
    print("Distribution de la cible :")
    vc = df[TARGET_COL].value_counts()
    display(pd.DataFrame({"count": vc, "%": (vc / len(df) * 100).round(2)}))
else:
    print("Distribution de la cible (régression) :")
    display(df[[TARGET_COL]].describe())
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    df[TARGET_COL].hist(bins=30, ax=axes[0])
    axes[0].set_title(f"Histogramme — {TARGET_COL}")
    sns.boxplot(y=df[TARGET_COL], ax=axes[1])
    axes[1].set_title(f"Boxplot — {TARGET_COL}")
    plt.tight_layout()
    plt.show()

## 5. Analyse exploratoire (EDA)

### 5.1 Distribution des features numériques

In [ ]:
if num_features:
    n = len(num_features)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flat

    for ax, feat in zip(axes, num_features):
        if TASK == "classification":
            for label, grp in df.groupby(TARGET_COL):
                sns.kdeplot(grp[feat], ax=ax, label=str(label), fill=True, alpha=0.3)
            ax.legend(fontsize=8)
        else:
            sns.histplot(df[feat], ax=ax, kde=True)
        ax.set_title(feat)

    for ax in list(axes)[n:]:
        ax.set_visible(False)

    title = "Distribution des features par classe" if TASK == "classification" else "Distribution des features"
    plt.suptitle(title, y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()

### 5.2 Heatmap des corrélations

In [ ]:
if num_features:
    corr_cols = num_features + ([TARGET_COL] if TASK == "regression" else [])
    corr = df[corr_cols].corr()
    plt.figure(figsize=(max(6, len(corr_cols)), max(5, len(corr_cols) - 1)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, linewidths=0.5)
    plt.title("Matrice de corrélation")
    plt.tight_layout()
    plt.show()

### 5.3 Pairplot

In [ ]:
# Limiter aux features les plus importantes si le dataset est large
MAX_PAIRPLOT_FEATURES = 6
pair_cols = num_features[:MAX_PAIRPLOT_FEATURES]

if pair_cols:
    hue = TARGET_COL if TASK == "classification" else None
    sns.pairplot(df[pair_cols + ([TARGET_COL] if hue else [])],
                 hue=hue, diag_kind="kde", plot_kws={"alpha": 0.5})
    plt.suptitle("Pairplot", y=1.01, fontsize=13)
    plt.show()

### 5.4 Boxplots par classe (classification)

In [ ]:
if TASK == "classification" and num_features:
    n = len(num_features)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flat

    for ax, feat in zip(axes, num_features):
        sns.boxplot(data=df, x=TARGET_COL, y=feat, ax=ax)
        ax.set_title(feat)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)

    for ax in list(axes)[n:]:
        ax.set_visible(False)

    plt.suptitle("Boxplots par classe", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()

### 5.5 Features catégorielles

In [ ]:
if cat_features:
    n = len(cat_features)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flat

    for ax, feat in zip(axes, cat_features):
        df[feat].value_counts().plot(kind="bar", ax=ax)
        ax.set_title(feat)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)

    for ax in list(axes)[n:]:
        ax.set_visible(False)

    plt.suptitle("Distribution des features catégorielles", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune feature catégorielle.")

## 6. Tableau récapitulatif

In [ ]:
if TASK == "classification" and num_features:
    summary = df.groupby(TARGET_COL)[num_features].agg(["mean", "std"]).round(2)
    summary.columns = [" ".join(col) for col in summary.columns]
    display(summary)
else:
    display(df[num_features].describe().round(2))

## 7. Conclusion

**Features les plus discriminantes :**
- …

**Corrélations :**
- …

**Qualité des données :**
- Valeurs manquantes : …
- Doublons : …
- Équilibre des classes : …

**Implications pour le ML :**
- Type de tâche : …
- Baseline attendue : …
- Points d'attention : …